In [3]:
import pandas as pd
import numpy as np

ModuleNotFoundError: No module named 'pandas'

In [ ]:
size_path = "D:\\ITB\\Thesis\\Preprocessing\\儲格設計_原檔(商品資訊).csv"
df_size = pd.read_csv(size_path, sep=';', decimal=',', encoding='utf-8-sig', engine = "python")

In [3]:
df_size.info()

<class 'pandas.DataFrame'>
RangeIndex: 10321 entries, 0 to 10320
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   item_code  10321 non-null  int64  
 1   註記         10321 non-null  str    
 2   (箱)長cm     9646 non-null   float64
 3   (箱)寬cm     9646 non-null   float64
 4   (箱)高cm     9646 non-null   float64
 5   (箱)重量      9646 non-null   float64
 6   箱入數        10055 non-null  float64
 7   板入數        10055 non-null  str    
 8   庫存量        9966 non-null   float64
 9   庫存單位       10321 non-null  str    
 10  庫存換算(箱)    9718 non-null   float64
 11  目前庫存(PCS)  10130 non-null  float64
dtypes: float64(8), int64(1), str(3)
memory usage: 967.7 KB


In [4]:
# Filter the SKU to certain size

cols = ['(箱)長cm', '(箱)寬cm', '(箱)高cm']
df_size[cols] = df_size[cols].apply(pd.to_numeric, errors='coerce')

# The longest dimension
df_size['highest_dimension'] = df_size[['(箱)長cm', '(箱)寬cm', '(箱)高cm']].max(axis=1)

# The second longest dimension
df_size['second_highest_dimension'] = df_size[['(箱)長cm', '(箱)寬cm', '(箱)高cm']].apply(lambda x: sorted(x)[-2], axis=1)

# The shortest dimension
df_size['shortest_dimension'] = df_size[['(箱)長cm', '(箱)寬cm', '(箱)高cm']].min(axis=1)

df_size = df_size[
    (df_size['highest_dimension'] <= 118) &
    (df_size['second_highest_dimension'] <= 48) &
    (df_size['shortest_dimension'] <= 45)
]

In [5]:
df_size.info()

<class 'pandas.DataFrame'>
Index: 9519 entries, 0 to 10319
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   item_code                 9519 non-null   int64  
 1   註記                        9519 non-null   str    
 2   (箱)長cm                    9519 non-null   float64
 3   (箱)寬cm                    9519 non-null   float64
 4   (箱)高cm                    9519 non-null   float64
 5   (箱)重量                     9519 non-null   float64
 6   箱入數                       9519 non-null   float64
 7   板入數                       9519 non-null   str    
 8   庫存量                       9195 non-null   float64
 9   庫存單位                      9519 non-null   str    
 10  庫存換算(箱)                   9195 non-null   float64
 11  目前庫存(PCS)                 9338 non-null   float64
 12  highest_dimension         9519 non-null   float64
 13  second_highest_dimension  9519 non-null   float64
 14  shortest_dimension     

In [6]:
df_size = df_size[
    (df_size['(箱)寬cm'] != 0) &
    (df_size['(箱)高cm'] != 0) &
    (df_size['(箱)長cm'] != 0)
].copy()


df_size["max_comp_number"] = df_size.apply(
    lambda row: (
        118 // row['highest_dimension'] *
        48 // row['second_highest_dimension'] *
        45 // row['shortest_dimension']
    ),
    axis=1
).round().astype(int)

In [9]:
df_size.head(20)

,item_code,(箱)長cm,(箱)寬cm,(箱)高cm,max_comp_number
0,10000001001,45.0,30.0,24.0,5
1,10000002001,34.0,23.0,31.0,7
2,10000003001,40.0,27.0,12.0,11
3,10000004001,34.0,23.0,31.0,7
4,10000005001,40.0,28.0,12.0,11
5,10000006001,40.0,13.0,13.0,24
6,10000007001,40.0,28.0,12.0,11
7,10000008001,37.0,26.0,28.0,8
8,10000010001,34.0,23.0,31.0,7
9,10000011001,40.0,27.0,10.0,13


In [8]:
# Save the result to a new CSV file
df_size = df_size.filter(items=['item_code', '(箱)長cm', '(箱)寬cm', '(箱)高cm', 'max_comp_number'])
df_size.to_csv("D:/ITB/Thesis/scratch_ma/max_comp_number.csv", index=False, encoding="utf-8-sig")